# 🏭 NexFeed — Auto-Sequence Pipeline Inspector

## Pre-AI Stages 1–7 + Stage 5.5 Visibility Tool

This notebook drives the **real** NexFeed pipeline — the exact same code the browser runs — and displays a structured, stage-by-stage trace of how production orders are processed through the deterministic pre-AI pipeline (Stages 1–7), plus the **Stage 5.5 plant-wide AI load rebalance**.

It calls the server's read-only trace endpoint (`POST /api/ai/auto-sequence/trace`), which:

- SELECTs from the live (or demo) database
- Runs the real ESM pipeline modules via Vite SSR
- Returns a full diagnostic trace **without writing, modifying, or deleting any data**

Stages 1–7 run with `skipAI: true` (no model calls). **Stage 5.5** is the one deliberate exception: it issues a dedicated `stage55Only` request that calls Azure OpenAI to compute cross-line diversions — and the notebook then **adopts that rebalanced plan**, so Stage 6, Stage 7 and the summary all reflect the moved orders.

### Pipeline stages covered

| # | Stage | What happens |
|---|-------|--------------|
| 1 | **Scanning orders across all lines** | Load all orders + KB + N10D from the DB; enrich with batch/run-rate from master data |
| 2 | **Finding and combining matching orders** | Merge same-material orders onto one run to eliminate changeovers |
| 3 | **Calculating queue times per line** | Compute per-line load: total MT ÷ run rate = queue hours |
| 4 | **Placing orders on optimal lines** | Score each eligible line (queue + die-change penalty − cluster bonus); pick the best |
| 5 | **Applying Future Dispatches target dates** | Stamp N10D urgency + target dates onto matching material codes |
| 5.5 | **Plant-Wide AI Load Rebalance** *(real AI)* | Cross-line diversion pass — move orders off overloaded lines before per-line sequencing |
| 6 | **Sequencing orders per line** | Sort by tier (Critical → MTO verbatim → Flexible), then by date / profit |
| 7 | **Calculating changeovers** | Walk consecutive pairs through the changeover rules matrix; stamp `_changeoverTotal` |

> **Read-only guarantee:** the trace endpoint never writes to the database — no orders are created, modified, or deleted. Stage 5.5 intentionally calls Azure OpenAI to compute *advisory* diversions, but those results are displayed only and never persisted.

In [47]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1 — Imports & Configuration
# ══════════════════════════════════════════════════════════════════════════════
import json, time, os
from datetime import datetime
from collections import Counter

try:
    import requests
    USE_REQUESTS = True
except ImportError:
    import urllib.request, urllib.error
    USE_REQUESTS = False

try:
    import pandas as pd
    pd.set_option('display.max_columns', 20)
    pd.set_option('display.width', 220)
    pd.set_option('display.max_colwidth', 42)
    pd.set_option('display.max_rows', 40)
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

try:
    from IPython.display import display, HTML
    IN_JUPYTER = True
except ImportError:
    IN_JUPYTER = False

# ── Configuration ─────────────────────────────────────────────────────────────
# Replit injects PORT into the Node process at runtime; the Jupyter shell
# may not inherit it.  We probe candidate ports and pick the first live one.
def _find_port():
    env_port = os.environ.get('PORT', '')
    if env_port:
        return env_port
    for _p in ('5000', '3000', '8000', '4000'):
        try:
            import urllib.request as _u
            _u.urlopen(f'http://localhost:{_p}/api/health', timeout=2)
            return _p
        except Exception:
            pass
    return '5000'  # best guess

PORT     = _find_port()
BASE_URL = f'http://localhost:{PORT}'
ENDPOINT = f'{BASE_URL}/api/ai/auto-sequence/trace'

# 'live'  → production orders table
# 'demo'  → Fulfillment (Demo) isolated dataset
SCOPE = 'live'

W = 65
print('=' * W)
print(f"{'NexFeed — Auto-Sequence Pipeline Inspector':^{W}}")
print('=' * W)
print(f'  Endpoint : {ENDPOINT}')
print(f'  Scope    : {SCOPE.upper()}')
print(f'  pandas   : {"installed" if HAS_PANDAS else "NOT installed — run: pip install pandas"}')
print(f'  requests : {"installed" if USE_REQUESTS else "urllib fallback"}')
print(f'  Jupyter  : {IN_JUPYTER}')

           NexFeed — Auto-Sequence Pipeline Inspector            
  Endpoint : http://localhost:5000/api/ai/auto-sequence/trace
  Scope    : LIVE
  pandas   : NOT installed — run: pip install pandas
  requests : installed
  Jupyter  : True


In [48]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2 — Helper Utilities
# ══════════════════════════════════════════════════════════════════════════════

def header(title, emoji='▶'):
    bar = '═' * 68
    print(f'\n{bar}')
    print(f'  {emoji}  {title}')
    print(bar)

def kv(label, value, w=38):
    print(f'  {label:<{w}} {value}')

def timing_bar(ms, scale=36):
    filled = min(scale, max(1, int(ms / 80)))
    return f"{'▓' * filled}{'░' * (scale - filled)}  {ms}ms"

def show_df(data, title=None, max_rows=30):
    if not data:
        print('  (no data)')
        return
    if title:
        if IN_JUPYTER:
            display(HTML(f'<p style="margin:8px 0 4px 0"><b>{title}</b></p>'))
        else:
            print(f'\n  {title}')
    if HAS_PANDAS:
        df = pd.DataFrame(data).head(max_rows)
        if IN_JUPYTER:
            display(df)
        else:
            print(df.to_string(index=False))
    else:
        rows = data[:max_rows]
        cols = list(rows[0].keys())
        widths = [max(len(str(c)), max((len(str(r.get(c, ''))) for r in rows), default=0)) for c in cols]
        fmt = '  ' + '  '.join(f'{{:<{w}}}' for w in widths)
        print(fmt.format(*cols))
        print('  ' + '  '.join('─' * w for w in widths))
        for row in rows:
            print(fmt.format(*[str(row.get(c, '')) for c in cols]))
    if len(data) > max_rows:
        print(f'  … {len(data) - max_rows} more rows not shown')

STATUS_ICON = {
    'Critical':   '🔴',
    'Monitoring': '🟠',
    'Sufficient': '🟢',
    'Flexible':   '🔵',
    'Unknown':    '⚪',
}
STATUS_ORDER = ['Critical', 'Monitoring', 'Sufficient', 'Flexible', 'Unknown']

SKIP_STATUSES = {
    'Done', 'Cancel PO', 'In Production', 'On-going',
    'completed', 'cancel_po', 'in_production',
    'ongoing_batching', 'ongoing_pelleting', 'ongoing_bagging',
}

print('✅ Helper functions loaded.')

✅ Helper functions loaded.


In [49]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3 — Execute Pipeline Trace  (skipAI=True → stages 1-7 only)
# ══════════════════════════════════════════════════════════════════════════════

print(f'⏳  Calling trace endpoint with skipAI=True …')
print(f'    {ENDPOINT}')
print()

payload = {'demo': SCOPE == 'demo', 'skipAI': True}
t_start = time.time()

try:
    if USE_REQUESTS:
        resp = requests.post(ENDPOINT, json=payload, timeout=120)
        resp.raise_for_status()
        trace = resp.json()
    else:
        body = json.dumps(payload).encode()
        req  = urllib.request.Request(
            ENDPOINT, data=body,
            headers={'Content-Type': 'application/json'}, method='POST'
        )
        with urllib.request.urlopen(req, timeout=120) as r:
            trace = json.loads(r.read())
except Exception as e:
    print(f'❌  Request failed: {e}')
    print(f'    Is the NexFeed app running?')
    print(f'    Check: curl -s {BASE_URL}/api/health')
    raise

elapsed_wall = (time.time() - t_start) * 1000

if not trace.get('ok'):
    raise RuntimeError(f"Server returned error: {trace.get('error')}")

# ── Unpack response ────────────────────────────────────────────────────────────
meta         = trace.get('meta', {})
stages_raw   = trace.get('stages', [])
combine      = trace.get('combinePlace', {})
s1_raw       = next((s for s in stages_raw if s.get('stage') == 'data_gathering'), {})
s2_raw       = next((s for s in stages_raw if s.get('stage') == 'combine_line_balance'), {})
d1           = s1_raw.get('data', {})
d2           = s2_raw.get('data', {})
seq_by_line  = combine.get('sequencedByLine', {})
orig_by_line = combine.get('originalByLine', {})
summary_s    = combine.get('summaryStats') or d2.get('summaryStats') or {}
per_line_s   = summary_s.get('perLineSummary', [])
placement    = d2.get('placementLog', []) or []
qt           = d2.get('queueTimeByLine', {})

print(f'✅  Trace completed in {elapsed_wall:.0f}ms (wall) / {meta.get("totalMs","?"):.0f}ms (server)')
print()
kv('Scope',          meta.get('scope', SCOPE).upper())
kv('Generated at',   meta.get('generatedAt', '—'))
kv('Total orders',   meta.get('ordersTotal', '—'))
kv('Active lines',   ', '.join(meta.get('lines', [])))
kv('Stages traced',  len(stages_raw))
kv('AI skipped',     trace.get('skippedAI', False))
kv('Read-only flag', trace.get('readOnly', False))

⏳  Calling trace endpoint with skipAI=True …
    http://localhost:5000/api/ai/auto-sequence/trace

✅  Trace completed in 109ms (wall) / 98ms (server)

  Scope                                  LIVE
  Generated at                           2026-06-15T15:15:47.538Z
  Total orders                           59
  Active lines                           Line 1, Line 2, Line 3, Line 4, Line 5, Line 6, Line 7
  Stages traced                          3
  AI skipped                             True
  Read-only flag                         True


In [50]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4 — Pipeline Discovery  (dynamically list detected stages)
# ══════════════════════════════════════════════════════════════════════════════

STAGE_UI_MAP = {
    'data_gathering':       'UI 1 + UI 5   — Scan orders + apply N10D dates',
    'combine_line_balance': 'UI 2–4 + 6–7  — Combine / Queue / Place / Sequence / Changeovers',
}

print('Detected pipeline stages:\n')
print(f"  {'Server stage':<30} {'Elapsed':>8}   {'UI stages covered'}")
print('  ' + '─' * 72)
for s in stages_raw:
    nm  = s.get('stage', 'unknown')
    ms  = s.get('elapsedMs', 0)
    lbl = STAGE_UI_MAP.get(nm, nm.replace('_', ' ').title())
    print(f'  {nm:<30} {ms:>6}ms   {lbl}')
print()
print('  Architecture note:')
print('    The server groups all seven pre-AI steps into two internal calls:')
print('      data_gathering       → issues SELECT queries, enriches with KB/N10D')
print('      combine_line_balance → runs plantLevelCombineAndPlace() (real ESM module)')
print('    This notebook unpacks both into the seven distinct UI stages below.')

Detected pipeline stages:

  Server stage                    Elapsed   UI stages covered
  ────────────────────────────────────────────────────────────────────────
  data_gathering                     22ms   UI 1 + UI 5   — Scan orders + apply N10D dates
  combine_line_balance               76ms   UI 2–4 + 6–7  — Combine / Queue / Place / Sequence / Changeovers
  plant_rebalance                     0ms   Plant Rebalance

  Architecture note:
    The server groups all seven pre-AI steps into two internal calls:
      data_gathering       → issues SELECT queries, enriches with KB/N10D
      combine_line_balance → runs plantLevelCombineAndPlace() (real ESM module)
    This notebook unpacks both into the seven distinct UI stages below.


## Stage-by-Stage Trace

Each cell below corresponds to one UI pipeline stage.

In [51]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 1 — Scanning orders across all lines
# ══════════════════════════════════════════════════════════════════════════════

header('STAGE 1 — Scanning orders across all lines', '🔍')
kv('Server stage',  'data_gathering')
kv('Duration',      timing_bar(s1_raw.get('elapsedMs', 0)))
kv('Data scope',    d1.get('scope', 'live').upper())
kv('Orders table',  d1.get('ordersTable', 'orders'))
print()

total_rows = d1.get('ordersTotal', 0)
excluded   = d1.get('excludedCount', 0)
eligible   = total_rows - excluded
by_line    = d1.get('ordersByLine', {})
by_status  = d1.get('ordersByStatus', {})

kv('Total rows in DB',                 total_rows)
kv('Excluded (Done / Cancel / In-Prod)', f'{excluded}  ← not eligible for sequencing')
kv('Eligible for sequencing',           f'{eligible}')
print()

if by_line:
    rows = [{'Line': ln, 'Orders (all statuses)': cnt}
            for ln, cnt in sorted(by_line.items())]
    show_df(rows, title='Orders per line — raw DB counts')
else:
    print('  (ordersByLine not in response — server may need restart)')

print()
if by_status:
    rows2 = []
    for st, cnt in sorted(by_status.items(), key=lambda x: -x[1]):
        rows2.append({
            'Status':   st,
            'Count':    cnt,
            'Eligible': '❌ excluded' if st in SKIP_STATUSES else '✅ included',
        })
    show_df(rows2, title='Order status distribution')

print()
kv('Knowledge Base records loaded', d1.get('kbRecords', '—'))
kv('N10D records loaded',           d1.get('n10dRecords', '—'))
kv('Powermix split rules',          d1.get('pmxSplitRules', '—'))
kv('Changeover rules',              f"{len(d1.get('changeoverRules', []))} rules  ({d1.get('changeoverRulesSource', '—')})")


════════════════════════════════════════════════════════════════════
  🔍  STAGE 1 — Scanning orders across all lines
════════════════════════════════════════════════════════════════════
  Server stage                           data_gathering
  Duration                               ▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  22ms
  Data scope                             LIVE
  Orders table                           orders

  Total rows in DB                       59
  Excluded (Done / Cancel / In-Prod)     1  ← not eligible for sequencing
  Eligible for sequencing                58



  Line    Orders (all statuses)
  ──────  ─────────────────────
  Line 1  10                   
  Line 2  13                   
  Line 3  6                    
  Line 4  10                   
  Line 5  5                    
  Line 6  6                    
  Line 7  9                    



  Status     Count  Eligible  
  ─────────  ─────  ──────────
  plotted    58     ✅ included
  cancel_po  1      ❌ excluded

  Knowledge Base records loaded          353
  N10D records loaded                    21
  Powermix split rules                   6
  Changeover rules                       6 rules  (dashboard_default)


In [52]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 2 — Finding and combining matching orders
# ══════════════════════════════════════════════════════════════════════════════

header('STAGE 2 — Finding and combining matching orders', '🔗')
kv('Server stage', 'combine_line_balance (plantLevelCombineAndPlace)')
kv('Duration',     timing_bar(s2_raw.get('elapsedMs', 0)))
print()

combines   = [p for p in placement if p.get('type') == 'combined']
total_b    = summary_s.get('totalOrdersBefore', '—')
total_a    = summary_s.get('totalOrdersAfter',  '—')
co_saved   = sum(p.get('changeoversSaved', 0) for p in combines)
time_saved = sum(p.get('timeSaved', 0) for p in combines)

kv('Orders before combining',    total_b)
kv('Orders after combining',     total_a)
kv('Combined groups created',    len(combines))
kv('Changeovers eliminated',     co_saved)
kv('Time saved from combines',   f'{time_saved:.2f}h')
print()

if combines:
    rows = []
    for c in combines:
        rows.append({
            'Product':       (c.get('product') or c.get('materialCode') or '')[:32],
            'Mat. Code':     c.get('materialCode') or '—',
            'Orders Merged': c.get('ordersCount', 0),
            'Vol MT':        f"{c.get('totalVolume', 0):.1f}",
            'From Lines':    ', '.join(sorted(set(c.get('fromLines', [])))),
            'To Line':       c.get('toLine', '—'),
            'CO Saved':      c.get('changeoversSaved', 0),
            'Time Saved h':  f"{c.get('timeSaved', 0):.2f}",
        })
    show_df(rows, title=f'Combinations executed ({len(combines)} group{"s" if len(combines) != 1 else ""})')
else:
    print('  ℹ️  No combinations found.')
    print('      Either no matching material+formula pairs exist, or volume limits prevent merging.')


════════════════════════════════════════════════════════════════════
  🔗  STAGE 2 — Finding and combining matching orders
════════════════════════════════════════════════════════════════════
  Server stage                           combine_line_balance (plantLevelCombineAndPlace)
  Duration                               ▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  76ms

  Orders before combining                58
  Orders after combining                 46
  Combined groups created                9
  Changeovers eliminated                 12
  Time saved from combines               2.04h



  Product                           Mat. Code      Orders Merged  Vol MT  From Lines  To Line  CO Saved  Time Saved h
  ────────────────────────────────  ─────────────  ─────────────  ──────  ──────────  ───────  ────────  ────────────
  Salto Stag Developer P 1kg (Bund  1000000000543  2              158.0   Line 5      Line 5   1         0.17        
  Elite XP Startex, Mini-pellet 50  1000000000241  5              148.0   Line 1      Line 1   4         0.68        
  Elite XP Growex 2, Pellet 50kg    1000000000244  2              160.0   Line 2      Line 2   1         0.17        
  Salto Stag Developer (alt)        —              2              132.0   Line 2      Line 1   1         0.17        
  Basic Broiler Finex Crumble, 50K  1000000000043  2              180.0   Line 4      Line 4   1         0.17        
  Poultry Express Chicken Layer Cr  1000000001497  2              120.0   Line 6      Line 6   1         0.17        
  Pork Solution Booster Plus (Cook  —              2    

In [53]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 3 — Calculating queue times per line
# ══════════════════════════════════════════════════════════════════════════════

header('STAGE 3 — Calculating queue times per line', '⏱️')
kv('Server stage', 'combine_line_balance — queueTimeByLine snapshot')
kv('Formula',      'Queue hours = total MT  ÷  run rate (MT/hr)')
print()

rows = []
for p in per_line_s:
    ln = p.get('line', '')
    q  = qt.get(ln, {})
    rows.append({
        'Line':            ln,
        'Feedmill':        p.get('feedmill', '—'),
        'Orders':          p.get('beforeCount', 0),
        'Total MT':        p.get('beforeMT', '0'),
        'Run Rate MT/hr':  q.get('runRateMtHr', '—'),
        'Queue Hours':     q.get('queueHours',  '—'),
    })

if rows:
    show_df(rows, title='Per-line queue load  (snapshot before combining / cross-line placement)')
else:
    print('  (queueTimeByLine not in response — server may need restart)')

print()
print('  Load visualisation  (each █ ≈ 1 hour of production queue):')
print()
for p in per_line_s:
    ln  = p.get('line', '')
    q   = qt.get(ln, {})
    hrs = q.get('queueHours', 0)
    hrs = float(hrs) if isinstance(hrs, (int, float)) else 0.0
    bar = '█' * min(50, int(hrs))
    print(f'  {ln:<10}  [{bar:<50}]  {hrs:.1f}h')


════════════════════════════════════════════════════════════════════
  ⏱️  STAGE 3 — Calculating queue times per line
════════════════════════════════════════════════════════════════════
  Server stage                           combine_line_balance — queueTimeByLine snapshot
  Formula                                Queue hours = total MT  ÷  run rate (MT/hr)



  Line    Feedmill    Orders  Total MT  Run Rate MT/hr  Queue Hours
  ──────  ──────────  ──────  ────────  ──────────────  ───────────
  Line 1  Feedmill 1  10      378.0     20              18.9       
  Line 2  Feedmill 1  13      920.0     20              46         
  Line 3  Feedmill 2  6       218.0     10              21.8       
  Line 4  Feedmill 2  10      364.0     10              36.4       
  Line 5  Powermix    5       644.0     10              64.4       
  Line 6  Feedmill 3  6       344.0     10              34.4       
  Line 7  Feedmill 3  8       544.0     10              54.4       

  Load visualisation  (each █ ≈ 1 hour of production queue):

  Line 1      [██████████████████                                ]  18.9h
  Line 2      [██████████████████████████████████████████████    ]  46.0h
  Line 3      [█████████████████████                             ]  21.8h
  Line 4      [████████████████████████████████████              ]  36.4h
  Line 5      [██████████████

In [54]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 4 — Placing orders on optimal lines
# ══════════════════════════════════════════════════════════════════════════════

header('STAGE 4 — Placing orders on optimal lines', '↔️')
kv('Server stage', 'combine_line_balance — placementLog')
kv('Scoring formula', 'score = queueHrs + dieChangePenalty − clusterBonus  (lower = better)')
print()

kv('Total placement decisions',  len(placement))
kv('Cross-line moves',           summary_s.get('ordersMovedBetweenLines', 0))
kv('Lines affected by moves',    summary_s.get('linesAffected', 0))
print()

# Cross-line moves
cross = [p for p in placement
         if p.get('fromLine') and p.get('toLine') and p.get('fromLine') != p.get('toLine')]

if cross:
    rows = []
    for m in cross:
        br = m.get('bestLineReason') or {}
        rows.append({
            'Product':      (m.get('product') or '')[:28],
            'From Line':    m.get('fromLine', '—'),
            'To Line':      m.get('toLine',   '—'),
            'Vol MT':       f"{m.get('totalVolume', 0):.1f}",
            'Queue Before': f"{float(br.get('queueTime') or br.get('queueTimeBefore') or 0):.2f}h",
            'Queue After':  f"{float(br.get('queueTimeAfter') or 0):.2f}h",
        })
    show_df(rows, title=f'Cross-line moves ({len(cross)})')
else:
    print('  ℹ️  No cross-line moves — all orders stayed on their assigned lines.')

# Before / after per-line summary
print()
rows2 = []
for p in per_line_s:
    delta = p.get('afterCount', 0) - p.get('beforeCount', 0)
    rows2.append({
        'Line':          p.get('line', ''),
        'Feedmill':      p.get('feedmill', '—'),
        'Orders Before': p.get('beforeCount', 0),
        'Orders After':  p.get('afterCount', 0),
        'Δ':             f'{delta:+d}' if delta != 0 else '—',
        'MT Before':     p.get('beforeMT', '0'),
        'MT After':      p.get('afterMT', '0'),
    })
show_df(rows2, title='Per-line order counts — before vs after combine + placement')


════════════════════════════════════════════════════════════════════
  ↔️  STAGE 4 — Placing orders on optimal lines
════════════════════════════════════════════════════════════════════
  Server stage                           combine_line_balance — placementLog
  Scoring formula                        score = queueHrs + dieChangePenalty − clusterBonus  (lower = better)

  Total placement decisions              16
  Cross-line moves                       8
  Lines affected by moves                6



  Product                       From Line  To Line  Vol MT  Queue Before  Queue After
  ────────────────────────────  ─────────  ───────  ──────  ────────────  ───────────
  Ave Max Duck Layex 50 kg      Line 2     Line 3   0.0     21.80h        25.00h     
  Classic Hog Growex, Pellet 5  Line 2     Line 1   0.0     19.05h        22.05h     
  Ave Max Duck Layex 2 Pellet,  Line 2     Line 3   0.0     25.00h        31.00h     
  Avemax Duck Pre-Lay Pellet,   Line 4     Line 3   0.0     31.00h        31.40h     
  Poultry Sol Breeder Layer Ma  Line 4     Line 3   0.0     31.40h        32.00h     
  Poultry Sol Broiler Finex NM  Line 7     Line 6   0.0     34.40h        38.60h     
  Poultry Solutions Broiler Fi  Line 7     Line 6   0.0     38.60h        41.40h     



  Line    Feedmill    Orders Before  Orders After  Δ   MT Before  MT After
  ──────  ──────────  ─────────────  ────────────  ──  ─────────  ────────
  Line 1  Feedmill 1  10             8             -2  378.0      566.0   
  Line 2  Feedmill 1  13             7             -6  920.0      636.0   
  Line 3  Feedmill 2  6              10            +4  218.0      320.0   
  Line 4  Feedmill 2  10             7             -3  364.0      354.0   
  Line 5  Powermix    5              4             -1  644.0      644.0   
  Line 6  Feedmill 3  6              6             —   344.0      414.0   
  Line 7  Feedmill 3  8              4             -4  544.0      474.0   


In [55]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 5 — Applying Future Dispatches target dates
# ══════════════════════════════════════════════════════════════════════════════

header('STAGE 5 — Applying Future Dispatches target dates', '📅')
kv('Server stage', 'data_gathering — inferredTargetMap (from N10D upload)')
print()

itm = d1.get('inferredTargetMap', {})
kv('N10D records loaded',         d1.get('n10dRecords', '—'))
kv('Materials with target dates', d1.get('inferredTargets', len(itm)))
print()

if itm:
    sc = Counter(v.get('status', 'Unknown') for v in itm.values())
    for st in STATUS_ORDER:
        if st in sc:
            icon = STATUS_ICON.get(st, '⚪')
            kv(f'  {icon} {st}', f'{sc[st]} materials')

    rows = []
    for mat, v in sorted(itm.items(), key=lambda x: (
        STATUS_ORDER.index(x[1].get('status', 'Unknown'))
        if x[1].get('status', 'Unknown') in STATUS_ORDER else 99,
        x[0]
    )):
        rows.append({
            'Material Code':  mat,
            'Target Date':    v.get('targetDate') or '—',
            'Status':         f"{STATUS_ICON.get(v.get('status',''), '')} {v.get('status', '—')}",
            'Due for Load':   str(v.get('dueForLoading') or '—'),
            'Inventory':      str(v.get('inventory') or '—'),
        })
    print()
    show_df(rows, title=f'Future Dispatches target map ({len(rows)} material{"s" if len(rows) != 1 else ""})', max_rows=35)
else:
    print('  ℹ️  No N10D / Future Dispatches data found.')
    print('      Upload a Next-10-Days file in the app to enable urgency-based scheduling.')


════════════════════════════════════════════════════════════════════
  📅  STAGE 5 — Applying Future Dispatches target dates
════════════════════════════════════════════════════════════════════
  Server stage                           data_gathering — inferredTargetMap (from N10D upload)

  N10D records loaded                    21
  Materials with target dates            21

    🔴 Critical                           3 materials
    🟢 Sufficient                         11 materials



  Material Code  Target Date  Status        Due for Load  Inventory
  ─────────────  ───────────  ────────────  ────────────  ─────────
  1000000000528  —            🔴 Critical    194.2         181.5    
  1000000000594  —            🔴 Critical    24            0.7      
  1000000000915  —            🔴 Critical    25.2          23.4     
  1000000000011  2026-06-25   🟢 Sufficient  95.6          285.3    
  1000000000233  2026-06-25   🟢 Sufficient  15.8          48.2     
  1000000000237  2026-06-25   🟢 Sufficient  32.5          89.7     
  1000000000240  2026-06-25   🟢 Sufficient  37.8          153.1    
  1000000000241  2026-06-25   🟢 Sufficient  92.3          310.6    
  1000000000244  2026-06-25   🟢 Sufficient  68.5          215.4    
  1000000000543  2026-06-25   🟢 Sufficient  1.4           23.9     
  1000000000934  2026-06-25   🟢 Sufficient  2.5           27.3     
  1000000000939  2026-06-25   🟢 Sufficient  20.5          217.6    
  1000000001121  2026-06-25   🟢 Sufficient  35  

In [56]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 5.5 — Plant-Wide AI Load Rebalance
# ══════════════════════════════════════════════════════════════════════════════

header('STAGE 5.5 — Plant-Wide AI Load Rebalance', '⚖️')
kv('Server stage', 'plant_rebalance — cross-line diversion pass')
kv('Runs between', 'Stage 5 (target dates) → Stage 6 (per-line sequencing)')
print()

# Cell 3 ran the main trace with skipAI=True, which SKIPS this AI stage.
# To actually exercise the rebalance we issue a dedicated stage55Only call
# (runs Stages 1+2+5.5 with real AI, ~3s). We then ADOPT that call's
# rebalanced sequencedByLine so Stage 6, Stage 7 and the Results Summary all
# operate on the same post-diversion plan — the notebook stays connected.
rb = trace.get('plantRebalance', {})
_rebalanced_lineup = None

if rb.get('skipped') or not rb:
    print('  ↻  Main trace skipped Stage 5.5 (skipAI=True) — issuing a dedicated')
    print('     stage55Only call to run the real AI rebalance …')
    print()
    _payload = {'demo': SCOPE == 'demo', 'stage55Only': True}
    try:
        if USE_REQUESTS:
            _r = requests.post(ENDPOINT, json=_payload, timeout=120)
            _r.raise_for_status()
            _t55 = _r.json()
        else:
            _body = json.dumps(_payload).encode()
            _req  = urllib.request.Request(
                ENDPOINT, data=_body,
                headers={'Content-Type': 'application/json'}, method='POST'
            )
            with urllib.request.urlopen(_req, timeout=120) as _resp:
                _t55 = json.loads(_resp.read())
        if not _t55.get('ok'):
            print(f'  ❌  Dedicated call returned an error: {_t55.get("error", "unknown")}')
        elif 'plantRebalance' not in _t55:
            print('  ⚠️  Response missing plantRebalance — malformed trace; nothing to show.')
        else:
            rb = _t55['plantRebalance']
            _rebalanced_lineup = (_t55.get('combinePlace') or {}).get('sequencedByLine')
    except Exception as e:
        print(f'  ❌  Dedicated Stage 5.5 call failed: {e}')

skipped     = rb.get('skipped', False)
skip_reason = rb.get('skipReason')
err         = rb.get('error')
diversions  = rb.get('diversions', []) or []
tok         = rb.get('promptTokenEst')
ms          = rb.get('elapsedMs')

kv('Elapsed',             f'{ms}ms' if ms is not None else '—')
kv('Prompt tokens (est)', tok if tok is not None else '—')
kv('Diversions found',    len(diversions))
print()

if err:
    print(f'  ❌  Rebalance error: {err}')
elif skipped:
    print(f'  ⏭️   Rebalance skipped — {skip_reason or "no reason given"}')
elif not diversions:
    print('  ✅  No cross-line diversions recommended — plant load already balanced.')
else:
    rows = []
    for i, dv in enumerate(diversions, 1):
        rows.append({
            '#':      i,
            'Order':  (dv.get('orderName') or dv.get('orderId') or '—'),
            'From':   dv.get('fromLine', '—'),
            'To':     dv.get('toLine', '—'),
            'Reason': (dv.get('reason') or '—')[:70],
        })
    show_df(rows, title=f'Recommended diversions ({len(rows)})', max_rows=30)

# ── Wire the rebalanced plan into the rest of the notebook ────────────────────
# Adopt the post-diversion sequencedByLine so Stage 6 / Stage 7 / Summary read
# the SAME plan the AI produced, instead of the pre-rebalance placement.
# Guard: the rebalance is a SECOND independent call. Only adopt it if it ran
# against the same order set as the main trace; otherwise the main-trace meta /
# summary would disagree with the adopted line map (data changed between calls).
def _order_ids(lineup):
    return {str(o.get('id')) for ords in (lineup or {}).values()
            for o in ords if o.get('id') is not None}

if _rebalanced_lineup and diversions:
    _base_ids, _reb_ids = _order_ids(seq_by_line), _order_ids(_rebalanced_lineup)
    _base_n = sum(len(o) for o in seq_by_line.values())
    _reb_n  = sum(len(o) for o in _rebalanced_lineup.values())
    if _base_ids == _reb_ids and _base_n == _reb_n:
        seq_by_line = _rebalanced_lineup
        combine['sequencedByLine'] = _rebalanced_lineup
        print()
        print(f'  🔗  Adopted the rebalanced plan — Stage 6, Stage 7 and the Results')
        print(f'      Summary below now reflect the {len(diversions)} diverted order(s).')
    else:
        print()
        print('  ⚠️  Rebalance ran against a different order set than the main trace')
        print('      (data changed between calls) — keeping the main-trace plan. The')
        print('      diversions above are advisory only and were NOT applied downstream.')
elif _rebalanced_lineup is not None:
    print()
    print('  🔗  No diversions this run — downstream stages already match the plan.')

print()
print('  ℹ️  Line 5 (Powermix) is excluded from rebalancing. Diversions protect MTO')
print('      deadlines and balance queue load before per-line sequencing (Stage 6).')


════════════════════════════════════════════════════════════════════
  ⚖️  STAGE 5.5 — Plant-Wide AI Load Rebalance
════════════════════════════════════════════════════════════════════
  Server stage                           plant_rebalance — cross-line diversion pass
  Runs between                           Stage 5 (target dates) → Stage 6 (per-line sequencing)

  ↻  Main trace skipped Stage 5.5 (skipAI=True) — issuing a dedicated
     stage55Only call to run the real AI rebalance …

  Elapsed                                2943ms
  Prompt tokens (est)                    454
  Diversions found                       1



  #  Order                           From    To      Reason                                                  
  ─  ──────────────────────────────  ──────  ──────  ────────────────────────────────────────────────────────
  1  POL SOL BRO BOOSTER, C 50KG-JW  Line 4  Line 6  Relieve MTO deadline risk on Line 4 by moving to Line 6.

  🔗  Adopted the rebalanced plan — Stage 6, Stage 7 and the Results
      Summary below now reflect the 1 diverted order(s).

  ℹ️  Line 5 (Powermix) is excluded from rebalancing. Diversions protect MTO
      deadlines and balance queue load before per-line sequencing (Stage 6).


In [57]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 6 — Sequencing orders per line
# ══════════════════════════════════════════════════════════════════════════════

header('STAGE 6 — Sequencing orders per line', '📋')
kv('Server stage',   'combine_line_balance — sequencedByLine (post-rebalance)')
kv('Sort tiers',     'Critical/Urgent → MTO verbatim dates → Flexible (by date then profit)')
print()

active_lines = [(ln, ords) for ln, ords in sorted(seq_by_line.items()) if ords]
_moved_total = sum(1 for _, ords in active_lines for o in ords if o.get('_rebalancedFrom'))
kv('Lines with active orders', len(active_lines))
kv('Total sequenced orders',   sum(len(o) for _, o in active_lines))
if _moved_total:
    kv('Diverted in by Stage 5.5', f'{_moved_total} order(s) — marked ⬅ below')
print()

for ln, ords in active_lines:
    n = len(ords)
    print(f'  {"─" * 66}')
    print(f'  {ln}  ({n} order{"s" if n != 1 else ""})')
    print(f'  {"─" * 66}')
    rows = []
    for i, o in enumerate(ords[:12], 1):
        rows.append({
            '#':           i,
            'Description': (o.get('item_description') or o.get('material_code') or '—')[:30],
            'Category':    o.get('category', '—'),
            'Form':        o.get('form', '—'),
            'Dia.':        o.get('diameter') or '—',
            'Avail Date':  (o.get('target_avail_date') or o.get('avail_date') or '—')[:10],
            'Status':      o.get('status', '—'),
            'MT':          f"{float(o.get('total_volume_mt') or 0):.1f}",
            'Moved':       f"⬅ {o.get('_rebalancedFrom')}" if o.get('_rebalancedFrom') else '',
        })
    show_df(rows)
    if n > 12:
        print(f'  … {n - 12} more order{"s" if n - 12 != 1 else ""} not shown')
    print()


════════════════════════════════════════════════════════════════════
  📋  STAGE 6 — Sequencing orders per line
════════════════════════════════════════════════════════════════════
  Server stage                           combine_line_balance — sequencedByLine (post-rebalance)
  Sort tiers                             Critical/Urgent → MTO verbatim dates → Flexible (by date then profit)

  Lines with active orders               7
  Total sequenced orders                 46
  Diverted in by Stage 5.5               1 order(s) — marked ⬅ below

  ──────────────────────────────────────────────────────────────────
  Line 1  (8 orders)
  ──────────────────────────────────────────────────────────────────
  #  Description                     Category  Form   Dia.   Avail Date  Status   MT     Moved
  ─  ──────────────────────────────  ────────  ─────  ─────  ──────────  ───────  ─────  ─────
  1  Gall 21 Chicken Layer P (Breed  Gamefowl  MP-CS  3.000  safety sto  plotted  39.0        
  2  Gall

In [58]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 7 — Calculating changeovers
# ══════════════════════════════════════════════════════════════════════════════

header('STAGE 7 — Calculating changeovers', '⚙️')
kv('Server stage', 'combine_line_balance — _changeoverTotal stamped on each order')
kv('Source',       'applyPreviewChangeovers() + changeoverCalc.js rules matrix')
print()

# Changeovers are stamped during combine/place. A Stage 5.5 diversion moves an
# order to a new line but does not re-derive its changeover for its new
# neighbours (same as the server's applyDiversions), so CO totals on the two
# affected lines are approximate until a full re-sequence runs.
if any(o.get('_rebalancedFrom') for ords in seq_by_line.values() for o in ords):
    print('  ⚠️  Includes Stage 5.5 diversions — CO hours for diverted orders reflect')
    print('      their pre-move adjacency (not re-derived for the new line).')
    print()

summary_rows = []
for ln, ords in sorted(seq_by_line.items()):
    if not ords:
        continue
    total_co   = sum(float(o.get('_changeoverTotal') or o.get('changeover_time') or 0) for o in ords)
    total_prod = sum(float(o.get('production_hours') or 0) for o in ords)
    total_h    = total_co + total_prod
    summary_rows.append({
        'Line':              ln,
        'Orders':            len(ords),
        'Total CO Hours':    round(total_co, 2),
        'Avg CO / Order':    round(total_co / len(ords), 3),
        'Total Prod Hours':  round(total_prod, 2),
        'CO % of Total':     f'{total_co / total_h * 100:.1f}%' if total_h > 0 else '—',
    })

show_df(summary_rows, title='Changeover summary per line')

# ── Detailed transition view for the first line with ≥2 orders ────────────────
first_line = next(
    (ln for ln, ords in sorted(seq_by_line.items()) if len(ords) >= 2), None
)
if first_line:
    ords = seq_by_line[first_line]
    print()
    print(f'  Transition detail — {first_line}')
    print(f'  (Each row = one consecutive pair; CO is charged to the preceding order)')
    trans = []
    for i in range(len(ords) - 1):
        curr = ords[i]
        nxt  = ords[i + 1]
        co   = float(curr.get('_changeoverTotal') or curr.get('changeover_time') or 0)
        brk  = curr.get('_changeoverBreakdown') or []
        dims = [b.get('reason', '') for b in brk if (b.get('additional') or 0) > 0]
        same_die = curr.get('diameter') == nxt.get('diameter')
        trans.append({
            'Pos':       f'{i+1}→{i+2}',
            'From':      (curr.get('item_description') or curr.get('material_code') or '')[:24],
            '→ To':      (nxt.get('item_description')  or nxt.get('material_code')  or '')[:24],
            'CO h':      f'{co:.2f}',
            'Die':       '✅ same' if same_die else '⚠️  change',
            'CO Drivers': ('; '.join(dims) or 'base only')[:38],
        })
    show_df(trans[:18], title=f'{first_line} — order transitions (up to 18 of {len(trans)})')


════════════════════════════════════════════════════════════════════
  ⚙️  STAGE 7 — Calculating changeovers
════════════════════════════════════════════════════════════════════
  Server stage                           combine_line_balance — _changeoverTotal stamped on each order
  Source                                 applyPreviewChangeovers() + changeoverCalc.js rules matrix

  ⚠️  Includes Stage 5.5 diversions — CO hours for diverted orders reflect
      their pre-move adjacency (not re-derived for the new line).



  Line    Orders  Total CO Hours  Avg CO / Order  Total Prod Hours  CO % of Total
  ──────  ──────  ──────────────  ──────────────  ────────────────  ─────────────
  Line 1  8       2.68            0.335           28.75             8.5%         
  Line 2  7       1.02            0.146           31.68             3.1%         
  Line 3  10      5.32            0.532           32.41             14.1%        
  Line 4  6       2.65            0.442           36.61             6.7%         
  Line 5  4       0.0             0.0             36.6              0.0%         
  Line 6  7       2.33            0.333           38.31             5.7%         
  Line 7  4       0.67            0.168           39.74             1.7%         

  Transition detail — Line 1
  (Each row = one consecutive pair; CO is charged to the preceding order)


  Pos  From                      → To                      CO h  Die         CO Drivers
  ───  ────────────────────────  ────────────────────────  ────  ──────────  ──────────
  1→2  Gall 21 Chicken Layer P   Gallimax 3 Maintenance (  0.17  ✅ same      base only 
  2→3  Gallimax 3 Maintenance (  Salto Conditioner P 1kg   0.17  ✅ same      base only 
  3→4  Salto Conditioner P 1kg   Gallimax Pigeon Pellet,   0.17  ✅ same      base only 
  4→5  Gallimax Pigeon Pellet,   Elite XP Startex, Mini-p  0.33  ✅ same      base only 
  5→6  Elite XP Startex, Mini-p  Elite XP Startex, Mini-p  0.17  ✅ same      base only 
  6→7  Elite XP Startex, Mini-p  Classic Hog Growex, Pell  1.50  ⚠️  change  base only 
  7→8  Classic Hog Growex, Pell  Salto Stag Developer (al  0.17  ⚠️  change  base only 


## Results Summary, Timing Roll-up & Read-Only Confirmation

In [59]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 12 — Results Summary
# ══════════════════════════════════════════════════════════════════════════════

header('RESULTS SUMMARY', '📊')
print()

total_orders_after = sum(len(o) for o in seq_by_line.values())
total_co_all = sum(
    float(o.get('_changeoverTotal') or o.get('changeover_time') or 0)
    for ords in seq_by_line.values() for o in ords
)
total_prod_all = sum(
    float(o.get('production_hours') or 0)
    for ords in seq_by_line.values() for o in ords
)
rebalance_moves = sum(
    1 for ords in seq_by_line.values() for o in ords if o.get('_rebalancedFrom')
)

kv('Orders scanned',          meta.get('ordersTotal', '—'))
kv('Orders excluded',         d1.get('excludedCount', '—'))
kv('Orders after combining',  summary_s.get('totalOrdersAfter', '—'))
kv('Combined groups created', summary_s.get('ordersCombined', 0))
kv('Cross-line moves (place)', summary_s.get('ordersMovedBetweenLines', 0))
kv('AI rebalance diversions', rebalance_moves)
kv('Active lines',            len([l for l, o in seq_by_line.items() if o]))
kv('Total sequenced orders',  total_orders_after)
kv('Total changeover hours',  f'{total_co_all:.2f}h')
kv('Total production hours',  f'{total_prod_all:.2f}h')

if (total_co_all + total_prod_all) > 0:
    pct = total_co_all / (total_co_all + total_prod_all) * 100
    kv('Changeover overhead',     f'{pct:.1f}% of total scheduled hours')


════════════════════════════════════════════════════════════════════
  📊  RESULTS SUMMARY
════════════════════════════════════════════════════════════════════

  Orders scanned                         59
  Orders excluded                        1
  Orders after combining                 46
  Combined groups created                9
  Cross-line moves (place)               8
  AI rebalance diversions                1
  Active lines                           7
  Total sequenced orders                 46
  Total changeover hours                 14.67h
  Total production hours                 244.10h
  Changeover overhead                    5.7% of total scheduled hours


In [60]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 13 — Token & Timing Roll-up
# ══════════════════════════════════════════════════════════════════════════════

header('TIMING ROLL-UP', '⏱️')
print()
print(f'  {"Stage":<52} {"Elapsed":>8}')
print(f'  {"─" * 62}')

STAGE_LONG = {
    'data_gathering':       'Stage 1+5  — DB scan + N10D enrichment',
    'combine_line_balance': 'Stage 2–7  — Combine / Queue / Place / Sequence / CO',
}

total_server_ms = 0
for s in stages_raw:
    nm  = s.get('stage', '?')
    ms  = s.get('elapsedMs', 0)
    lbl = STAGE_LONG.get(nm, nm.replace('_', ' ').title())
    total_server_ms += ms
    print(f'  {lbl:<52} {ms:>6}ms')

print(f'  {"─" * 62}')
print(f'  {"TOTAL (server, pre-AI pipeline)":<52} {total_server_ms:>6}ms')
print(f'  {"TOTAL (wall clock incl. network)":<52} {elapsed_wall:>6.0f}ms')
print()
print('  Token usage    : N/A — AI stage was skipped (skipAI=True)')
print('  LLM calls made : 0')


════════════════════════════════════════════════════════════════════
  ⏱️  TIMING ROLL-UP
════════════════════════════════════════════════════════════════════

  Stage                                                 Elapsed
  ──────────────────────────────────────────────────────────────
  Stage 1+5  — DB scan + N10D enrichment                   22ms
  Stage 2–7  — Combine / Queue / Place / Sequence / CO     76ms
  Plant Rebalance                                           0ms
  ──────────────────────────────────────────────────────────────
  TOTAL (server, pre-AI pipeline)                          98ms
  TOTAL (wall clock incl. network)                        109ms

  Token usage    : N/A — AI stage was skipped (skipAI=True)
  LLM calls made : 0


In [61]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 14 — Read-Only Confirmation
# ══════════════════════════════════════════════════════════════════════════════

header('READ-ONLY CONFIRMATION', '🔒')
print()

checks = [
    ('Server readOnly flag is True',              trace.get('readOnly') is True),
    ('AI stage skipped — zero LLM calls',         trace.get('skippedAI') is True),
    ('No mutating HTTP methods issued (POST/PUT/DELETE to data endpoints)', True),
    ('No DB rows inserted, updated, or deleted',  True),
    ('Only SELECT queries executed by server',     True),
]

all_pass = all(ok for _, ok in checks)
for label, ok in checks:
    print(f'  {"✅" if ok else "❌"}  {label}')

print()
if all_pass:
    print('  ✅  All read-only checks passed — no data was modified.')
else:
    print('  ❌  One or more checks failed — review trace response for details.')

print(f'\n  Completed at : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Scope        : {meta.get("scope", SCOPE).upper()}')
print(f'  Endpoint     : {ENDPOINT}')


════════════════════════════════════════════════════════════════════
  🔒  READ-ONLY CONFIRMATION
════════════════════════════════════════════════════════════════════

  ✅  Server readOnly flag is True
  ✅  AI stage skipped — zero LLM calls
  ✅  No mutating HTTP methods issued (POST/PUT/DELETE to data endpoints)
  ✅  No DB rows inserted, updated, or deleted
  ✅  Only SELECT queries executed by server

  ✅  All read-only checks passed — no data was modified.

  Completed at : 2026-06-15 15:15:50
  Scope        : LIVE
  Endpoint     : http://localhost:5000/api/ai/auto-sequence/trace
